# Model Registry, Versioning & Governance

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/ml-in-practice/18-model-registry-and-governance

We build a tiny model registry with promotion stages, a promotion gate, lineage, and one-step rollback.

Self-contained: NumPy + matplotlib only. No torch, no sklearn, no network, no API keys.

> **To save your work:** click **Copy to Drive** at the top, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark style matching the site theme.
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.edgecolor': '#475569',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlecolor': '#e2e8f0',
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'grid.color': '#2e3347',
    'savefig.facecolor': '#0f1117',
})
BRAND = '#6366f1'
TEAL = '#14b8a6'
ROSE = '#f43f5e'
YELLOW = '#eab308'

rng = np.random.default_rng(0)

## 1. A minimal registry

A registry is a versioned catalogue. Each version carries **lineage** (code commit, data version, config) and a **metric**, and lives in a **stage**: None → Staging → Production → Archived.

In [ ]:
class Registry:
    def __init__(self):
        self.versions = {}   # version -> record
        self.production = None
        self._n = 0
    def register(self, metric, code, data, config):
        self._n += 1
        v = f'v{self._n}'
        self.versions[v] = {'metric': metric, 'stage': 'Staging',
                            'lineage': {'code': code, 'data': data, 'config': config}}
        return v
    def promote(self, v):
        if self.production:
            self.versions[self.production]['stage'] = 'Archived'
        self.versions[v]['stage'] = 'Production'
        self.production = v
    def serving(self):
        return self.production

reg = Registry()
v1 = reg.register(0.81, 'abc123', 'data@2026-01', {'lr': 0.01, 'seed': 0})
reg.promote(v1)
print('serving:', reg.serving(), '->', reg.versions[v1]['lineage'])

## 2. Promotion gate + rollback

A new candidate is promoted only if it beats the current production model on a held-out metric and clears a floor. Rollback is a one-line stage change — re-point Production at the previous version.

In [ ]:
def gated_promote(reg, candidate_metric, floor, **lineage):
    prod_metric = reg.versions[reg.production]['metric'] if reg.production else -1
    v = reg.register(candidate_metric, **lineage)
    if candidate_metric > prod_metric and candidate_metric >= floor:
        reg.promote(v); print(f'{v} PROMOTED ({candidate_metric:.3f} > {prod_metric:.3f})')
    else:
        print(f'{v} rejected ({candidate_metric:.3f} vs prod {prod_metric:.3f}, floor {floor})')
    return v

v2 = gated_promote(reg, 0.85, 0.80, code='def456', data='data@2026-02', config={'lr':0.01,'seed':0})
v3 = gated_promote(reg, 0.79, 0.80, code='ghi789', data='data@2026-03', config={'lr':0.02,'seed':0})
print('serving now:', reg.serving())

# Rollback: a bad deploy of v2 -> re-point Production at v1
def rollback(reg, to_version):
    reg.versions[reg.production]['stage'] = 'Archived'
    reg.versions[to_version]['stage'] = 'Production'
    reg.production = to_version
rollback(reg, v1)
print('after rollback, serving:', reg.serving())

## ✏️ Your turn — the audit query

Implement `who_served(reg, version)` returning the lineage triplet (code, data, config) for a given version — the core of an audit trail ('which model decided this, trained on what?').

In [ ]:
def who_served(reg, version):
    """TODO(you): return reg.versions[version]['lineage']."""
    # TODO
    return ...


In [ ]:
lin = who_served(reg, v2)
print('v2 lineage:', lin)
assert lin['code'] == 'def456'
assert lin['data'] == 'data@2026-02'
assert lin['config']['lr'] == 0.01
print('✅ every version is reproducible and auditable from its lineage.')

<details>
<summary>Solution</summary>

```python
def who_served(reg, version):
    return reg.versions[version]['lineage']
```

Capture lineage **at registration time** — code commit, data snapshot, config/seed, metrics. Reconstructing it after an incident is painful or impossible.
</details>

## Recap

- A **registry** versions models across **stages**; deployment serves whatever is in Production.
- A **promotion gate** (beat prod + clear a floor) is where evaluation and sign-off attach.
- **Rollback** is a stage change, not a rebuild.
- **Lineage** (code, data, config, metrics) makes a version reproducible and auditable — capture it at registration.